# COMP30850 - Exam Spring 2025/26

Imports needed for the whole notebook

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import itertools
from networkx import community
from collections import Counter
import numpy as np

---
## Task 1

#### Task 1(a)
a) Load the two CSV files. From these files, create a co-starring network where
actors are connected if they appeared in the same movie. Include edge
weights representing the number of movies in which both actors appeared.
Report the number of nodes and edges in the network.

First and foremost, loading both csv files in pandas dataframes

In [2]:
actors_df = pd.read_csv('actors.csv')
movies_df = pd.read_csv('movies.csv')

Viewing the head of each df to see how the data is formatted

In [3]:
actors_df.head()

,name,gender
0,Aaron Taylor-Johnson,male
1,Adam Driver,male
2,Allison Janney,female
3,America Ferrera,female
4,Amy Adams,female


In [4]:
movies_df.head()

,title,actor
0,12 Years a Slave,Michael Fassbender
1,12 Years a Slave,Lupita Nyong'o
2,12 Years a Slave,Benedict Cumberbatch
3,12 Years a Slave,Paul Dano
4,12 Years a Slave,Sarah Paulson


In [5]:
# mapping movie title to list of actors in that movie

movie_to_actors = {}
for _, row in movies_df.iterrows():
    movie_title = row['title']
    actor_name = row['actor']
    if movie_title not in movie_to_actors:
        movie_to_actors[movie_title] = []
    movie_to_actors[movie_title].append(actor_name)

Verifying by indexing Avengers: Age of Ultron to see if the cast members are returned correctly and formatted in a list

In [6]:
# verify
movie_to_actors['Avengers: Age of Ultron']

['Robert Downey Jr.',
 'Chris Hemsworth',
 'Mark Ruffalo',
 'Chris Evans',
 'Scarlett Johansson',
 'Jeremy Renner',
 'Samuel L. Jackson',
 'Don Cheadle',
 'Aaron Taylor-Johnson']

Perfect! Now I can make a graph from this new dictionary.  
First up - initialising the graph, and adding a node for every actor

In [7]:
G = nx.Graph()

# add nodes
for _, row in actors_df.iterrows():
    G.add_node(row['name'])

Now that nodes are added, I can use `itertools.combinations()` to cleanly populate the graph with edges between actors who have appeared in the same movie(s) together.

In [8]:
# add edges with weights
for movie_title, actors_in_movie in movie_to_actors.items():
    for actor1, actor2 in itertools.combinations(actors_in_movie, r=2):
        if G.has_edge(actor1, actor2):
            G[actor1][actor2]['weight'] += 1
        else:
            G.add_edge(actor1, actor2, weight=1)

And finally reporting the number of nodes and edges

In [9]:
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

print(f'Graph has {num_nodes} nodes')
print(f'Graph has {num_edges} edges')

Graph has 116 nodes
Graph has 726 edges


#### Task 1(b)
b) Add a gender attribute to nodes in the network, based on the input data.
Report the gender distribution of the nodes.

In [10]:
# add gender attribute to nodes
for _, row in actors_df.iterrows():
    G.add_node(row['name'], gender=row['gender'])

Having looped over `actors_df`, I'll just quickly check to make sure the nodes were correctly attributed with the actors genders

In [11]:
# retrieve gender for michael fassbender from network to verify
G.nodes['Michael Fassbender']['gender']

'male'

That checks out, so now I'll calculate the distribution

In [12]:
network_genders = {'Female': 0, 'Male': 0}

for node in G.nodes():
    gender = G.nodes[node]['gender']
    if gender == 'female':
        network_genders['Female'] += 1
    elif gender == 'male':
        network_genders['Male'] += 1

In [14]:
prop_female = network_genders['Female'] / num_nodes
prop_male = network_genders['Male'] / num_nodes

print(f"Full Network Gender Proportion:")
print(f" Female: {prop_female:.2f}")
print(f" Male: {prop_male:.2f}")

Full Network Gender Proportion:
 Female: 0.28
 Male: 0.72


#### Task 1(c)
c) Identify the top 10 pairs of actors by edge weight.

In [15]:
# https://stackoverflow.com/questions/42856659/how-to-sort-edges-in-networkx-based-on-their-weight
edges = sorted(G.edges(data=True), key=lambda edge: edge[2].get('weight', 1), reverse=True)

# print the top 10 pairs of actors by edge weight
for i, edge in enumerate(edges[:10]):
    print(f"Pair {i+1}: {edge[0]} <-> {edge[1]}, Weight: {edge[2]['weight']}")

Pair 1: Chris Evans <-> Scarlett Johansson, Weight: 6
Pair 2: Robert Downey Jr. <-> Scarlett Johansson, Weight: 6
Pair 3: Chris Evans <-> Robert Downey Jr., Weight: 5
Pair 4: Chris Hemsworth <-> Mark Ruffalo, Weight: 5
Pair 5: Don Cheadle <-> Robert Downey Jr., Weight: 5
Pair 6: Don Cheadle <-> Scarlett Johansson, Weight: 5
Pair 7: Amy Adams <-> Henry Cavill, Weight: 4
Pair 8: Chris Evans <-> Chris Hemsworth, Weight: 4
Pair 9: Chris Evans <-> Mark Ruffalo, Weight: 4
Pair 10: Chris Evans <-> Jeremy Renner, Weight: 4


--- 
## Task 2


#### Task 2(a)
a) Apply a community finding algorithm to the co-starring network. Report the
number of communities detected.

Unsure of which community finding algorithm to choose for this co-starring network, I took inspiration from [this report](https://github.com/AndyKuKu/Louvain-Girvan-Newman-Which-one-is-better-for-a-large-social-network-), as well as a previous lab solution notebook, and chose to use Louvain algorithm for community finding.


In [18]:
communities = community.louvain_communities(G)
num_communities = len(communities)
print(f"Number of communities detected: {num_communities}")

Number of communities detected: 7


#### Task 2(b)
b) Calculate the proportion of nodes assigned to each community. How evenly is the network split across communities?

In [19]:
community_sizes = [len(c) for c in communities]
total_nodes = G.number_of_nodes()
proportions = [size / total_nodes for size in community_sizes]

print("Proportion of nodes in each community:")
for i, prop in enumerate(proportions):
    print(f"Community {i+1}: {prop:.4f}")

Proportion of nodes in each community:
Community 1: 0.1034
Community 2: 0.1121
Community 3: 0.2155
Community 4: 0.1466
Community 5: 0.0948
Community 6: 0.1897
Community 7: 0.1379


The network is not evenly split across communities. Community 3 is the largest, containing 21.55% of the nodes, while Community 5 is the smallest with 9.48%. The proportions vary significantly, indicating an uneven distribution.

In [21]:
# std deviation of community sizes
std_dev_proportions = np.std(proportions)
print(f"Standard deviation of community proportions: {std_dev_proportions:.4f}")

Standard deviation of community proportions: 0.0419


#### Task 2(c)
c) Create a subgraph corresponding to the largest community. Calculate the
density of this subgraph and compare it to the density of the full network.
What does this suggest about collaboration patterns within the community?

To create a subgraph of the largest community, I'll use networkx's built in `subgraph()` function, and gather the nodes in the subgraph from the largest community

In [23]:
largest_community_nodes = max(communities, key=len)
subgraph_largest_community = G.subgraph(largest_community_nodes)

density_largest_community = nx.density(subgraph_largest_community)
density_full_network = nx.density(G)

print(f"Density of the largest community subgraph: {density_largest_community:.4f}")
print(f"Density of the full network: {density_full_network:.4f}")

Density of the largest community subgraph: 0.2267
Density of the full network: 0.1088


The largest community subgraph is more than twice as dense as the full network, indicating a much higher concentration of connections within that community compared to the overall network.

--- 
## Task 3

#### Task 3(a)
a) Identify the top 10 actors in the network by weighted degree centrality.

In [24]:
weighted_degree_centrality = G.degree(weight='weight')
sorted_actors_by_weighted_degree = sorted(weighted_degree_centrality, key=lambda x: x[1], reverse=True)
top_10_actors_weighted_degree = sorted_actors_by_weighted_degree[:10]

for actor, degree in top_10_actors_weighted_degree:
    print(f"{actor}: {degree}")

Jennifer Lawrence: 67
Chris Hemsworth: 61
Scarlett Johansson: 54
Mark Ruffalo: 44
Chris Evans: 43
Robert Downey Jr.: 42
Stanley Tucci: 40
Woody Harrelson: 38
Dave Bautista: 36
Donald Sutherland: 36


#### Task 3(b)
b) Identify the female actor with the highest weighted degree centrality. Extract
the ego network for this actor and report how many nodes are in the ego
network and what percentage of the full network's nodes this represents. 

I know here I could just hard-code `highest_weighted_degree_female_actor='Jennifer Lawrence'` after looking at the output of the previous cell, but I want to make things a little more robust and tolerant of different data. So I first make a sublist from the `actors_df` that contains only actors where `gender` is female. Once I have that list I can loop through the list of sorted actors by weighted degree, and as soon as I land on one that's in my `female_actors_list` I can `break` from the loop.

In [25]:
female_actors_list = actors_df[actors_df['gender'] == 'female']['name'].tolist()

highest_weighted_degree_female_actor = None
max_weighted_degree = -1

for actor, degree in sorted_actors_by_weighted_degree:
    if actor in female_actors_list:
        max_weighted_degree = degree
        highest_weighted_degree_female_actor = actor
        break 

Now I've got `highest_weighted_degree_female_actor`, I can initialise an ego network around them. I'll calculate then what percentage of the full networks nodes are present in the ego graph.

In [26]:
ego_network = nx.ego_graph(G, highest_weighted_degree_female_actor)

num_nodes_ego_network = ego_network.number_of_nodes()
percentage_of_full_network = (num_nodes_ego_network / num_nodes) * 100

In [27]:
print(f"Number of nodes in {highest_weighted_degree_female_actor}'s ego network: {num_nodes_ego_network}")
print(f"This represents {percentage_of_full_network:.2f}% of the full network's nodes.")

Number of nodes in Jennifer Lawrence's ego network: 33
This represents 28.45% of the full network's nodes.


28.45% means that Jennifer Lawrence is directly connected to nearly a third of all actors in the co occurrence network.

#### Task 3(c)
c) Calculate the proportion of female and male actors in the ego network.
Compare this to the gender distribution of the full network.

In [32]:
ego_network_genders = {'Female': 0, 'Male': 0}

In [33]:
for node in ego_network.nodes():
    gender = ego_network.nodes[node]['gender']
    if gender == 'female':
        ego_network_genders['Female'] += 1
    elif gender == 'male':
        ego_network_genders['Male'] += 1

In [34]:
prop_female_ego = ego_network_genders['Female'] / num_nodes_ego_network
prop_male_ego = ego_network_genders['Male'] / num_nodes_ego_network

In [35]:
print(f"Ego Network Gender Proportion:")
print(f" Female: {prop_female_ego:.2f}")
print(f" Male: {prop_male_ego:.2f}")

print(f"Full Network Gender Proportion:")
print(f" Female: {prop_female:.2f}")
print(f" Male: {prop_male:.2f}")

Ego Network Gender Proportion:
 Female: 0.21
 Male: 0.79
Full Network Gender Proportion:
 Female: 0.28
 Male: 0.72


The ego network has a lower proportion of female actors (0.21) and a higher proportion of male actors (0.79) compared to the full network (0.28 female, 0.72 male). This suggests the ego network is more male-dominated than the overall network.